# functional-module-wrap — worked example 1: Wrap F.gelu in an nn.Module

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `functional-module-wrap`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

An `nn.Module` whose `forward` simply delegates to a stateless `torch.nn.functional` op is the pattern behind `nn.GELU`, `nn.ReLU`, and friends. Wrapping the functional makes it composable in `nn.Sequential`, visible in `model.modules()`, and movable with `.to(device)` — even though it holds no parameters.

## Worked solution

We build a `MyGELU` module that behaves exactly like `nn.GELU`.

1. Subclass `nn.Module` and call `super().__init__()` so the module machinery (the `_modules`, `_parameters` dicts) is initialized. Skipping this breaks `state_dict` and `.to()`.
2. There is no learnable state — GELU has no parameters — so `__init__` does nothing else.
3. `forward(x)` returns `F.gelu(x)` directly. We deliberately call the functional rather than instantiating `nn.GELU`, because the whole point is to exercise the functional-wrap pattern.
4. We confirm the wrap matches the built-in `nn.GELU` numerically and that `list(model.parameters())` is empty.

In [ ]:
import torch as t
import torch.nn as nn
import torch.nn.functional as F

t.manual_seed(0)

class MyGELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return F.gelu(x)

x = t.randn(5)
m = MyGELU()
print('matches nn.GELU:', bool(t.allclose(m(x), nn.GELU()(x))))
print('num params:', sum(1 for _ in m.parameters()))